# NeuralSWE: official 31B development comparison
Compare the frozen baseline and general repository-contract/cleanup prompt on two
deterministically selected public issues: rich_3067 and requests_6592. Each arm uses
the same model, sampling and per-task budgets. The order is AB on Rich and BA on
Requests. This is a preliminary development comparison, with one stochastic trial
per arm/case. These cases must be excluded from future candidate held-out studies.
No reference solutions or grader-specific identifiers are embedded in prompts.

Use the organizer's three inputs, original 2026-07-01 environment, L4 x4 and
Internet off. Run cells in order. Preserve failures and stop the GPU session after
exporting evidence. This does not submit a competition entry or establish a
leaderboard score, mobile memory benefit or general superiority.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, base64, zipfile, io, subprocess, importlib, glob
from datetime import datetime, timezone
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_OFFICIAL_SMOKE = True
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
if not IS_KAGGLE: print('Local syntax/configuration check only; no official inference executed')


In [ ]:
if IS_KAGGLE and RUN_OFFICIAL_SMOKE:
    if not WHEELHOUSE_DIR.is_dir() or not MODEL_PATH.is_dir():
        raise RuntimeError('Attach the organizer wheelhouse and exact model before execution')
    for key, value in {'LITELLM_LOCAL_MODEL_COST_MAP':'True', 'TRANSFORMERS_NO_TF':'1',
        'VLLM_WORKER_MULTIPROC_METHOD':'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS':'1',
        'VLLM_ENGINE_READY_TIMEOUT_S':'1200', 'VLLM_NO_USAGE_STATS':'1',
        'OTEL_SDK_DISABLED':'true', 'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}.items():
        os.environ[key] = value
    # Organizer workaround for broken cutlass hooks, restricted to that package's .pth files.
    for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for path in glob.glob(pattern): Path(path).unlink(missing_ok=True)
    wheel_dir = Path('/tmp/neural_swe_wheelhouse')
    wheel_dir.mkdir(exist_ok=True)
    for wheel in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in wheel.name.lower(): continue
        name = wheel.name.replace('cu128', '+cu128') if 'cu128' in wheel.name and '+' not in wheel.name else wheel.name
        link = wheel_dir / name
        if not link.exists(): link.symlink_to(wheel)
    wheels = sorted(str(p) for p in wheel_dir.glob('*.whl'))
    if not wheels: raise RuntimeError('Empty organizer wheelhouse')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                    '--no-deps', '--force-reinstall', *wheels], check=True)
    importlib.invalidate_caches()


In [ ]:
PLAN = json.loads('{"created_utc": "2026-10-03T15:04:25.237116+00:00", "scope": "public_development_not_held_out_or_leaderboard", "tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "selection": "minimum SHA256 of fixed salt and task ID within each declared repository; no reference or result screening", "tasks": [{"instance_id": "rich_3067", "repo": "Textualize/rich", "base_commit": "1ad6f73a4d1d3b40ac3f879feccfaad66412bd1d"}, {"instance_id": "requests_6592", "repo": "psf/requests", "base_commit": "0b4d494192de489701d3a2e32acef8fb5d3f042e"}], "arms": {"baseline": {"files": {"agent.yaml": "846e6e75c4845e790ba45f0b42884ac2d152b6c35368f546363bc3e6b4d666af", "prompts/system.md": "c7e1d29c42effae4c8b4f69c7cfaaceb13e8aa68c9f38ab492a20928bd1039b1", "configs/sampling.yaml": "6a66f5cd9328777d828e7111662e1d4923ca49e68161ebcde816184c3f2bc807"}}, "contract_cleanup_v2": {"files": {"agent.yaml": "846e6e75c4845e790ba45f0b42884ac2d152b6c35368f546363bc3e6b4d666af", "prompts/system.md": "cb96e0b69425dd395de0d2a5dcc21d5e9edf5eeee17bf40ac0cb2b7928c39481", "configs/sampling.yaml": "6a66f5cd9328777d828e7111662e1d4923ca49e68161ebcde816184c3f2bc807"}}}, "schedule": [{"task_id": "rich_3067", "arm": "baseline"}, {"task_id": "rich_3067", "arm": "contract_cleanup_v2"}, {"task_id": "requests_6592", "arm": "contract_cleanup_v2"}, {"task_id": "requests_6592", "arm": "baseline"}], "limits": {"timeout_seconds": 300, "max_time_minutes": 30, "max_tool_calls": 50, "max_turns": 100}, "same_model_and_backend": "official 31B QAT model revision 2 on L4 x4", "same_sampling": true, "sampling_seed_fixed": false, "repetitions_per_arm_per_case": 1, "reference_patches_exposed_to_agent": false, "exclude_from_future_candidate_held_out_studies": ["rich_3067", "requests_6592"], "existing_20261003_v1_study": "Preserved unchanged; this comparison does not evaluate or claim the locked memory-study effect."}')
PACKAGES = json.loads('{"baseline": {"sha256": "41c4f0261db4a7a01a33d24751f54263ec94b0f5d62b62c54d20bf70e8f1f85c", "base64": "UEsDBBQAAAAIAAAAPl3QPpAhogIAADoFAAAKAAAAYWdlbnQueWFtbK1Uy27bMBC86yu2yCUBIjUv5GCgBdzGbdGHU9gtehTW0komyofCXdpJT/2IfmG/pEupyOMeX6ilyNHM7KwP4NVz/ooDwJ68VHfoLPz9/QeWlCLa9Y/FDFYhCMzza3gbfGd6Pf2enEO4gCvakQ0DxfsDbiAxYoKHddo4w6yPeuG56X4LwUJLjcWI+WsMXbA27EG2BCH26M0vZfVhvlou1ut6tZhffVlUrj2GSNjC2cnZZXl6Up6cVxnsyZ1GNRirD63pOoqKHIMbcdPAotcdzK8+wcf19RK42ZLD6tkFFh4dzcCPXah5T/XYn6IlbqIZsuIZvC4A5kmCDy4kBg6d7DESkO+NJ4rG91NblTwKeNyZHoV4VGjpVteWNsjEx4rUGux90AI2qf8vWtuXCB59lI8BfQttxE4YnPHGoQXB2JNQCwOKGsKVwr1JxgpoDLJxU5qu1xMd0wDGZmuEGknKV28JRW3hYdgwxR2Vg0VfYiPlTlV0d2WkvHNUFYVTynYGfc5feVGen25KI+UNSrm/wNPLspGc5RYHRZyBQ+NrGyICHMB3r7pdtkPJa3s/h9UcOnWqlOSzV4dft2oGXB4VirG+YyF1wGvLUzNFzAZsVeVoDd1mzireYfzZhr1XKEvFo/MzeGF8Y1OrCvXGIPySR1CNYTHOkKcpvOpA1H4rIBf9tEt1E7wo2bzqzD0CmzYUDLWLynsc2pGyFpbKqeW5r/qWqmlUcp4YcjqUn55jVTbNSteZxqiOLUZPzDnL75K9r0fR2Z1I2U3W9SaZqDZsqAsxD0U+URWin+GZdr6EmHydvdaoTLWOXD3akytqjTxU+6g5eCg1RzULSuKx5PwfIvWYq2mDcnRqNs7o6Nc5wff3clF7Mv12EyI/3VagPuKwLf4BUEsDBBQAAAAIAAAAPl3kckGPVAEAAPACAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1srVLLjhMxELzPV5Q0Z8K+CLtBe+aCFiTEedTj6cy0Mn7I3V4STnwEX8iXYC8QCc4pyfLBdrke3ePxkuh6uBj2MutrJZ9WCfPmRH7Fz+8/8J4DZzKJAZ8ok2fjrPXFpRV8CVYCT1CjbFUBlK3tusFTRCJzyystzrEqxKccn9lzMCykGJkDPJOWzNPm4uI6Y59aCJV+h6vNNc7o8flPYsjsSYIixDC1kLwEUROHr2JLLAbCXo7NIfPUWUxDamQPb/AvejwVt3JR/C3jpYcDc4KPaoh7VPsjjbKKneBJ9YXusMPdFf5Hjw/ixfAcHY1lpXyCpsw0YR9zXa5o1TSfW+48HYcqNxUbLB446A7X29v7u0b1Mc8U5BtnzEUmBh+bRH4Hi0ZrmyLjo52juL15u73vbJFwqDaG30O26wAJbi0TDy2YebH6heXC9eB8dyzTzNYcPWy7X1BLAwQUAAAACAAAAD5d+vMwHkEGAAC5DAAAEQAAAHByb21wdHMvc3lzdGVtLm1kbVdNj9w2DL0bmP/AWxPAO4v2mJySJkV7aFGkLYqeFhqZHqsrS64+Znb+fR8pe9Ypetpd2yKfHh8fuX/FSiYx/cI1Gf/bn5+P9IVz9BemMjHluize8UA5juUqH7qcK9PpRjy44sKZXMmUeInZlZhuh86EgazxXt7leppdeVpMsdOR/shMMfibRl5SvLgBkUuMPh/p18SZE9Iu9eSdJRtDScaW3CK6efE8cygN1owEnAu+kueFaXQvCjamgpiAl2NNlknOFnyZj4fu0P1WTCry1k5snwXhmctTLqbU3O9uQdsjoPybbSEXcknVFhcDnkrQU3V+OHQzFzOYYo70+UXxKj5+WXBKgPBkLi6mnkbjlJLXJxIFKVMcWuBDhzsLqchBY4qzhlK+j/SDw9f8IgnAWihudJwyXV2ZKJ3pzRmRyI1Ug7kgkzl5fosrf2EzaJjEni8G9N2JNJKo11IhUv/KFCCOEZVeC4zSmJzxuqXvYgLRo/J9m0/RUzAzxBIps0l2espuRv70ZOPA76EOpFY9ZZCKsAMFvCCeTzwMCJ/7LsQCDZ4c+AP1AcxDig/ehHM1Z6Z/KifHuennnMwyUWB3nk5RCJg46AVP9Uw2xZw5dzMohTze0wUnx9t6CDiqx/XM2Ug5ydaURFCj8xL9ZzAtBZL7FeO3VFLcQzdE4BecS+KLHIoJ4AXuqjNU3MeknB7pwyW6gbzL2iCC7jpFzzuBqRo/Iw+0nicyd3nsBdEuN7LJDuU80k+t8K1/UgQaa2peNT7xofvw6093feGJKTTXLAWYcWPtFQDUUKqLJqeNhsIvZV/6J+EFOIX1a3KF9QHhPeD+jbgiQVSTr8og6fOBvTsxeJDL6inADlm6QVEPbhwP3Ri9j1e5riQC95+ikgugUYxHVNjTlc0z7g/xcVobbzJpeLCqn63DTMhX0e9Bqi7kQOa+aiHQtWMyrXFr4l4guhXM5AawSfjWDe1j7WIpyyeXLWAkBQyGqvEKiVINAY+1NTdjQMuO7ly3yn+pTY1jtKgM/CExOhPCQgIpU0CPwH5an40u5XKkjykawaKelJtDopNPdYA56SngWXjlHA6LBDOOxYpXPoJFVHbGd6sdFDdzrKWp5+4GO4flF3Rlc7h3NMSDNiAIYVUhmBILmMRwDFrdWuAfKzjAy8dJhLO4BXUOaG+Rn/wlB4VeaxahWjFusNZU2iLOuuJvKHjrjarih7OEi0sxiC09Zi512ZA0ss2eRxzmhQMYs3AFHV5oSulocbJh6wbK+OUUX2CgFMdR4Mplm85q6ICa1kMCfIjX4KPczdhnuA5E+fukHjCg5Egt3pWl7KXpThWh1iEZu690QG9csL6KvdFyW6VyLHH2ik6+lePH5fb2vU4uxduMZOvfI5T4vShiN6LI4Fzae02/PjI0swnIJ4WSpmp+3l6+ihz63kUTO0rQYZnAUTH5+Rt49CrWpr5ejAslFkmpLcrnLaiW6dgJSdIsQqMYDBcdYJt6Efjb72jC1TK9ctJKbNSuSU5K8nxUq2nDap3jGxrJ/36rHgyhYuL870Xw2McTGrblP2IANs1LBBlSW/+rLnU7ufsSnGoUMZoFFTNWBsbCcLIB/ShyFI+xE6ZSAzndloifGVesefN5cUOWvSZY3ubAUEX3Yon7ISFO8wOEpw2MyScSiTLnenpmzHK03oRLIXNzIHWHJTqZPbVk6daNqm3zeiPeDFOXCU+PZV4eg251T3DIp9fzx3l4+243P/p1EIiYH9eZ7qM1zXMPnXo0zWiE/j9OszOSHpuHhU9j0JRbv3odBoqxqzHyQxLraCrHTqcianSi1z7s7wdGDdaAizu3bjJu6LXy2nN3eoXBj21ejS7ASVCE/l7gM6BJVenhQUMrpPvDHz9/+NRv86btpDxDREY3WIwboT3TDduxTSqCdifZlr7aakVTq+nt54W+618lD4+FMtYWXxeOr8NwmrFQ4IbNDLaZoTS1Bq0pvJN9Smf6iTuAS3JzGU/YoVeq744AoLvGF23vhp3TdlyX537rBJnfIwddN9oVtM4udLthZKwVm2PftmTZbRw+XKdXMsj1C8v4tN64eR11i0zxQaNBvYduN1bw91KLSABtp9uMDHTgjwF9ue1dd4Ttf4lD9y9QSwECFAAUAAAACAAAAD5d0D6QIaICAAA6BQAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFtbFBLAQIUABQAAAAIAAAAPl3kckGPVAEAAPACAAAVAAAAAAAAAAAAAACkgcoCAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAAD5d+vMwHkEGAAC5DAAAEQAAAAAAAAAAAAAApIFRBAAAcHJvbXB0cy9zeXN0ZW0ubWRQSwUGAAAAAAMAAwC6AAAAwQoAAAAA"}, "contract_cleanup_v2": {"sha256": "83b9cf1543330a4a6f240e7634efc7665512491e87df8506c69a02ad483e5898", "base64": "UEsDBBQAAAAIAAAAPl3QPpAhogIAADoFAAAKAAAAYWdlbnQueWFtbK1Uy27bMBC86yu2yCUBIjUv5GCgBdzGbdGHU9gtehTW0komyofCXdpJT/2IfmG/pEupyOMeX6ilyNHM7KwP4NVz/ooDwJ68VHfoLPz9/QeWlCLa9Y/FDFYhCMzza3gbfGd6Pf2enEO4gCvakQ0DxfsDbiAxYoKHddo4w6yPeuG56X4LwUJLjcWI+WsMXbA27EG2BCH26M0vZfVhvlou1ut6tZhffVlUrj2GSNjC2cnZZXl6Up6cVxnsyZ1GNRirD63pOoqKHIMbcdPAotcdzK8+wcf19RK42ZLD6tkFFh4dzcCPXah5T/XYn6IlbqIZsuIZvC4A5kmCDy4kBg6d7DESkO+NJ4rG91NblTwKeNyZHoV4VGjpVteWNsjEx4rUGux90AI2qf8vWtuXCB59lI8BfQttxE4YnPHGoQXB2JNQCwOKGsKVwr1JxgpoDLJxU5qu1xMd0wDGZmuEGknKV28JRW3hYdgwxR2Vg0VfYiPlTlV0d2WkvHNUFYVTynYGfc5feVGen25KI+UNSrm/wNPLspGc5RYHRZyBQ+NrGyICHMB3r7pdtkPJa3s/h9UcOnWqlOSzV4dft2oGXB4VirG+YyF1wGvLUzNFzAZsVeVoDd1mzireYfzZhr1XKEvFo/MzeGF8Y1OrCvXGIPySR1CNYTHOkKcpvOpA1H4rIBf9tEt1E7wo2bzqzD0CmzYUDLWLynsc2pGyFpbKqeW5r/qWqmlUcp4YcjqUn55jVTbNSteZxqiOLUZPzDnL75K9r0fR2Z1I2U3W9SaZqDZsqAsxD0U+URWin+GZdr6EmHydvdaoTLWOXD3akytqjTxU+6g5eCg1RzULSuKx5PwfIvWYq2mDcnRqNs7o6Nc5wff3clF7Mv12EyI/3VagPuKwLf4BUEsDBBQAAAAIAAAAPl3kckGPVAEAAPACAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1srVLLjhMxELzPV5Q0Z8K+CLtBe+aCFiTEedTj6cy0Mn7I3V4STnwEX8iXYC8QCc4pyfLBdrke3ePxkuh6uBj2MutrJZ9WCfPmRH7Fz+8/8J4DZzKJAZ8ok2fjrPXFpRV8CVYCT1CjbFUBlK3tusFTRCJzyystzrEqxKccn9lzMCykGJkDPJOWzNPm4uI6Y59aCJV+h6vNNc7o8flPYsjsSYIixDC1kLwEUROHr2JLLAbCXo7NIfPUWUxDamQPb/AvejwVt3JR/C3jpYcDc4KPaoh7VPsjjbKKneBJ9YXusMPdFf5Hjw/ixfAcHY1lpXyCpsw0YR9zXa5o1TSfW+48HYcqNxUbLB446A7X29v7u0b1Mc8U5BtnzEUmBh+bRH4Hi0ZrmyLjo52juL15u73vbJFwqDaG30O26wAJbi0TDy2YebH6heXC9eB8dyzTzNYcPWy7X1BLAwQUAAAACAAAAD5dkxizItIHAACuEAAAEQAAAHByb21wdHMvc3lzdGVtLm1kbVfLjtw2ELwPMP/AW2xAM4vkaJ/8RAwkhuE8jJwWHKo1opciFT7m8fepblIzWiOnnZVEsru6urr4TyhKR1KfqUTt/vj2Ya++UgruRCqPpFKZZ2epVykM+cwf2pQKqcNVUW+z9Udlc1KR5pBsDvG63WjfK6Od43epHCabH2edzbhXfyVSwbur7DzHcLI9ds4huLRXXyIlijh2LgdnjTLB56hNTnVHO82OJvK5hjXhAEoZX/HzTGqwFwk2xIw9EV4KJRpSvDbjy7TfbrabP7KOmd+akcwTR3ik/JiyziV1qyzU8ghRfieTlfUpx2KyDR5PedNDsa7fbibKutdZ79WHi8Qr8dFlxioOhEZ9siF2atBWILk/4V1wZAx93Xi7Qc4MKs5QQwyTbCV479VHi6/pwgcANZ/tYCkmdbZ5VPGoXhyxk7KDKl6fcJI+OHqJlL+S7mWbSI5OGvDdgNR8UCelwk7dHSmEOARUuhUYpdEp4XU9frsJEUgPAvh1OgSnvJ7AlqAS6WjGx2QnBBAfTejpNeiBs4VQCahi3155vFA0HajvsX/qthsfMlh4sEAQ4HtgDzLunPbHoo+k/i0ULaXKoGPU86g82eN4CAzBSF5SPJSjMjGkRODMBFTBkNfqhKXDta1CJMUhQ33UXFFlSozMqcE63v53gM014gyzdstZXN/tpg/IgAOdI514UYgIn+NtVEPRXYgC6169OQXbK2eT9AiHdx6DoxXHhJAfcA7onkalbwxZc6JmN5BOFhXdq0+19rWFYkA0RpfUaD7SdvPmy6cbxfBEZzWVxCWYkLG0CwKUrYQalVELDJkueV39R8YFcTLs52gzyQOF9wj3O/ZlFqKedBYElTzvydkDAQdOVlYhbJ+4ISTq3g7DdjME58KZ0+WDgP37IOAi0MDaw0Ts1Jn0E/IH/yi23ht17HdGGLQ0mfbpzBSWqjM4YLorUgg07hB17d0SqeMQbQtmtD3QVPjW9vVjaWQuy3ubDMKIEjAQKtpJSCoW7/FYunPRBnTtYI9lqfzXUuk4BIPKQCIioTlBLBzAZfLoEihQbbXBxpT36m0MmmMRWUpVJNHMh9JDn2QV4pmpYQ6RxQETloWCVy4ARVR2wndNEbKdKJRc2XMThJXI0gV9WUXulepD7UAAQsJCIMUqMLLmaDS7MYh/KMAALx9GJs5sZ9TZo8GZfvwfL2R4jZ4ZaolxCasdJS1ijc3uioLX3ihCfoiLP9kYPCvTQ6Jc5iWSCrZe44jFNJMHYgayIPMLTckdzWLWL92gEn4cwgUaqsIwcLicbOVZgd4ibNVWceR9OHsXODltnqA7YOWfo4hAj5rjbJavxHXPlXhCCdEOPlL0+84E9cJ64wpLnJqvjSz7HCYn8fG3vH4/X1++lvElEVcpWTpYyPiOSbEaVEpjYVzLTdceaTWR9jiQa8V9VVW9vrzzHLuudmNFiqBiHgFT1unpJwh142slYMfahSozq0QZ+fO6qVQK+zFO3DCMJIsMZZljC4Ox88+/qBHJJXVHpZZZi2YrXsmnp6Y3dWi1eb7EwxG8XkoIVSgYPP+bCh67cEDX1gBkElbm8xY8rBYVEHaKTbmpE/RqYErqGVXThsfGTNCzHl3JpGSlMSOGU41yvM4BfxOSLGlRe9ZEYoPjDS3ToC/MfhbG9ajgEn8E+6SNMQCZJoHHXaeeCEMdDTgiK5xcdUg0Yg6WJ1DJiXt2wWqxYC9YoSHtPOrVQ57mBy/27hE6+Xhfv5/6l69WU6Rr44AZ/dBmuwtGV+XdbkSp1YRu6H7Qm5WcdLAgBmqNcZOvXVM8jBVtmjzSLrKAVKbD3AmPKpxouDfr/ICohhs42WPtKG37TkovjXeDlxF8W6fWYD30BEXobgU+IjSuqtrtZGsJ6fbw1w9v3nfL1KnmlCawSIuVxdBh2JO6wiabKCSoObFtemZvmVNN+tZTQ951d9JDacGM1ubNdjzfhuIEW4EMqyAsk0Ngqj1aon/Fvkom+wEYIrrIqfOUgptuWN9UAZGump/JvZp5Vjqy2ehuaQUe4wN5cR01Bym0hWiuhpI2hsWOXDXM7HEsvmxTLGoc9pl4jBqn7dRG3szTvJftwN/tZjVe8P9cMpMAjSeuhgc7Eggenbn4r1uI9VrB1f96d+7LvUECFEx2Nf4eM93OMgS2m28yGMV/qgGEZSiACTPWrrwKeF0N3DnEpwGGBW9Pt45ASKbwwCLR/toQnarWk10xr+Qw2pUGX8arMPvmsc0YgmiGfuai4OKQV7NEOJLNmeZaBccU/MHEQ/Uj1+BAzQzW+9EPlwtOAWT7u5b+R9HAKRnztfUZx7xYhWf3Gcu9kR/e/fZJpp0gjSnJnbG+esD4ejBAp7UYVKvAguJulrm1cB2fXrgSKUP9ThzcgLz4LdIu9VoEbJptu1/q3klfcn2RjJz+zEEnE+2cU+PDRRoUZoqNA9yTTM3iLa4YopTw+TaKAoMKa31l+HIkiHmTmnXXPlebNo12OxFuCOdKgvhd91x+boLk0q6aid2OB0rkH3Rh4SBe53tYXzSHMbB9mS00RB6IXZt6Vt9TpChPxC4mj6y3d2kTnVjhJKex+nGBtpu7o1/Bx1a73VWbsb1fs9eckabn1n3OuqY0tJjazCaQorFpdTu+jXS4dVY52IB6aZSqVwNoeYj/B1BLAQIUABQAAAAIAAAAPl3QPpAhogIAADoFAAAKAAAAAAAAAAAAAACkgQAAAABhZ2VudC55YW1sUEsBAhQAFAAAAAgAAAA+XeRyQY9UAQAA8AIAABUAAAAAAAAAAAAAAKSBygIAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUABQAAAAIAAAAPl2TGLMi0gcAAK4QAAARAAAAAAAAAAAAAACkgVEEAABwcm9tcHRzL3N5c3RlbS5tZFBLBQYAAAAAAwADALoAAABSDAAAAAA="}}')
if IS_KAGGLE and RUN_OFFICIAL_SMOKE:
    RUN = Path('/kaggle/working') / ('neural_swe_development_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    (RUN / 'plan.json').write_text(json.dumps(PLAN, indent=2) + '\n', encoding='utf-8')
    for arm, package in PACKAGES.items():
        payload = base64.b64decode(package['base64'], validate=True)
        if hashlib.sha256(payload).hexdigest() != package['sha256']:
            raise ValueError('Submission package mismatch: ' + arm)
        agent_dir = RUN / 'agents' / arm
        agent_dir.mkdir(parents=True)
        (RUN / (arm + '.zip')).write_bytes(payload)
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            if set(archive.namelist()) != {'agent.yaml','prompts/system.md','configs/sampling.yaml'}:
                raise ValueError('Unexpected submission contents')
            archive.extractall(agent_dir)
    print('Locked development plan:', RUN / 'plan.json')


In [ ]:
"""One real development issue using the organizer's published starter API.

API provenance: Ryan Holbrook, Getting Started - Gemma 4 Developer Agent,
version 2 (353282641), accessed 2026-10-03, Apache-2.0.
https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent
No substitute model, mock grader or inferred leaderboard score is provided.
"""
from __future__ import annotations

from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path
import time


def run(data_dir: Path, model_path: Path, agent_dir: Path, results_dir: Path, task_id: str,
        *, runtime: dict | None = None) -> dict:
    # Imports and API names below are from the organizer starter, not guesses.
    import asyncio
    import concurrent.futures
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    from swegemma.models.discovery import validate_single_declared_model

    model = 'gemma-4-31b-it-qat-w4a16-ct'
    if not model_path.is_dir() or not (data_dir / 'snapshots').is_dir():
        raise ValueError('Attach the exact required model and complete official data')
    if not torch.cuda.is_available() or torch.cuda.device_count() != 4:
        raise ValueError('This smoke study requires the competition L4 x4 environment')
    gpu_names = [torch.cuda.get_device_name(i) for i in range(4)]
    if any('L4' not in name for name in gpu_names):
        raise ValueError('Unexpected GPU environment; no model/backend substitution')
    tasks = [task for task in load_tasks(data_dir / 'tasks.jsonl') if task.instance_id == task_id]
    if len(tasks) != 1: raise ValueError('Select exactly one real development task')
    declared = validate_single_declared_model(agent_dir)
    if declared != model: raise ValueError('Declared base model differs from competition requirement')
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    # discover_adapters returns a manifest object, whose truth value does not
    # establish whether it contains adapters. This baseline is confined to the
    # three audited declarative files, so no adapter weights can enter the run.
    baseline_files = {'agent.yaml', 'prompts/system.md', 'configs/sampling.yaml'}
    actual_files = {p.relative_to(agent_dir).as_posix() for p in agent_dir.rglob('*') if p.is_file()}
    if actual_files != baseline_files:
        raise ValueError('Unexpected files in the no-adapter baseline')
    results_dir.mkdir(parents=True, exist_ok=False)
    report = {'scope': 'single_public_development_issue_not_leaderboard_or_mobile_result',
              'created_utc': datetime.now(timezone.utc).isoformat(), 'assigned_ids': [task_id],
              'model': model, 'model_path': str(model_path), 'gpu_names': gpu_names,
              'guide_sha256': hashlib.sha256((data_dir / 'HARNESS_README.md').read_bytes()).hexdigest(),
              'source_sha256': hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest(),
              'packages': {name: importlib.metadata.version(name) for name in
                           ('swegemma', 'adk-submission', 'adk-eval-core', 'vllm')},
              'assigned_count': 1, 'verified_resolved': 0, 'status': 'started',
              'official_leaderboard_score': None}
    report_path = results_dir / 'smoke_outcome.json'
    def save(): report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    save()
    started = time.monotonic()
    try:
        litellm.drop_params = True
        if runtime is not None and runtime:
            if runtime['model_path'] != str(model_path):
                raise ValueError('Cannot reuse a runtime with another model')
            models = runtime['models']
        else:
            server = VllmServer(VllmConfig(model=str(model_path), port=8000, host='127.0.0.1',
            tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
            dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
            gpu_memory_utilization=.80, enable_auto_tool_choice=True, enable_lora=True,
            max_loras=8, max_lora_rank=128, tensor_parallel_size=4, startup_timeout=1200),
            adapter_manifest=adapters)
            server.start()
            models = server.create_model_registry(aliases=[declared], model_prefix='openai/', api_key='EMPTY')
            if runtime is not None:
                runtime.update(model_path=str(model_path), models=models, server=server)
        limits, constraints = build_submission_limits()
        config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl', snapshots_dir=data_dir / 'snapshots',
            results_dir=results_dir, submission_dir=agent_dir, models=models, sandbox='subprocess',
            timeout_seconds=300, max_time_minutes=30, max_tool_calls=50, max_turns=100,
            limits=limits, generation_constraints=constraints, adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
            events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                token_threshold=14336, event_retention_size=5),
            graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'),
            wheels_dir=data_dir / 'wheels', verbose=False)
        evaluator = Evaluator(config)
        # A worker owns its asyncio loop, including inside a live notebook kernel.
        def evaluate():
            return asyncio.run(evaluator.evaluate_task(task=tasks[0], task_index=1, total_tasks=1))
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            result = pool.submit(evaluate).result()
        patch = result.agent_patch or ''
        (results_dir / 'generated.patch').write_text(patch, encoding='utf-8')
        (results_dir / 'test_output.txt').write_text(result.test_output or '', encoding='utf-8')
        # TaskResult is the organizer's Pydantic result type. Preserve grader
        # diagnostics rather than retaining only an exit code and losing cause.
        (results_dir / 'official_task_result.json').write_text(
            json.dumps(result.model_dump(mode='json', exclude={'trace'}), indent=2) + '\n', encoding='utf-8')
        report.update(status='completed', official_resolved=result.resolved,
                      test_exit_code=result.test_exit_code, tool_calls=result.tool_calls,
                      task_evaluation_duration_seconds=result.duration_seconds, patch_bytes=len(patch.encode()),
                      patch_sha256=hashlib.sha256(patch.encode()).hexdigest(),
                      verified_resolved=int(result.resolved is True and result.test_exit_code == 0 and bool(patch)))
    except Exception as error:
        report.update(status='failed', error_type=type(error).__name__, error=str(error))
        raise
    finally:
        report['setup_and_evaluation_wall_seconds'] = time.monotonic() - started
        save()
    return report


In [ ]:
"""A small, locked development comparison using the actual organizer evaluator.

Development results cannot be reported as held-out or leaderboard results. Every
assigned arm/task pair remains in the denominator, including interrupted runs.
"""
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path


def summarize(plan: dict, observations: list[dict]) -> dict:
    expected = {(row['arm'], row['task_id']) for row in plan['schedule']}
    if len(expected) != len(plan['schedule']):
        raise ValueError('Duplicate planned arm/task pair')
    seen, counts = set(), {arm: 0 for arm in plan['arms']}
    for row in observations:
        key = (row['arm'], row['task_id'])
        if key not in expected or key in seen:
            raise ValueError('Unexpected or duplicate observed arm/task pair')
        seen.add(key)
        result = row.get('result', {})
        passed = (result.get('status') == 'completed'
                  and result.get('official_resolved') is True
                  and result.get('test_exit_code') == 0
                  and result.get('patch_bytes', 0) > 0)
        counts[row['arm']] += int(passed)
    return {
        'scope': 'public_development_comparison_not_held_out_or_leaderboard',
        'assigned_pairs': len(expected), 'observed_pairs': len(seen),
        'unattempted_pairs': len(expected - seen),
        'arms': {arm: {'verified_resolved': counts[arm],
                      'assigned_count': sum(pair[0] == arm for pair in expected)}
                 for arm in plan['arms']},
        'official_leaderboard_score': None,
        'general_superiority_established': False,
        'interpretation': 'Two cases and one stochastic trial per arm cannot establish a general effect.',
    }


def run_development(data_dir: Path, model_path: Path, run_root: Path, plan: dict) -> dict:
    # The notebook embeds the audited official_smoke.run definition first.
    from competition.official_smoke import run
    return execute_plan(run, data_dir, model_path, run_root, plan)


def execute_plan(runner, data_dir: Path, model_path: Path, run_root: Path, plan: dict,
                 *, checkpoint_exporter=None) -> dict:
    destination = run_root / 'development_comparison.json'
    if destination.exists():
        raise FileExistsError('A comparison already exists in this run; preserve it and choose a new run directory')
    if hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest() != plan['tasks_sha256']:
        raise ValueError('Official task revision differs from the locked plan')
    for arm, identity in plan['arms'].items():
        agent_dir = run_root / 'agents' / arm
        observed = {p.relative_to(agent_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in agent_dir.rglob('*') if p.is_file()}
        if observed != identity['files']:
            raise ValueError('Agent configuration differs from locked plan: ' + arm)
    observations, runtime = [], {}
    def save():
        report = {'created_utc': datetime.now(timezone.utc).isoformat(),
                  'plan_sha256': hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
                  'observations': observations, **summarize(plan, observations)}
        destination.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        return report
    save()
    for index, assignment in enumerate(plan['schedule']):
        arm, task_id = assignment['arm'], assignment['task_id']
        print(f'DEVELOPMENT_START {index + 1}/{len(plan["schedule"])} {arm} {task_id}', flush=True)
        try:
            result = runner(data_dir, model_path, run_root / 'agents' / arm,
                            run_root / 'results' / f'{index:02d}_{arm}_{task_id}', task_id,
                            runtime=runtime)
            observations.append({'arm': arm, 'task_id': task_id, 'result': result})
            report = save()
            print('DEVELOPMENT_RESULT ' + json.dumps(observations[-1]), flush=True)
        except Exception as error:
            observations.append({'arm': arm, 'task_id': task_id,
                                 'result': {'status': 'infrastructure_exception',
                                            'error_type': type(error).__name__, 'error': str(error)}})
            save()
            if checkpoint_exporter is not None:
                checkpoint_exporter(run_root)
            # Preserve unattempted slots; do not keep generating on a broken backend.
            raise
        if checkpoint_exporter is not None:
            checkpoint_exporter(run_root)
    return report


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
from pathlib import Path
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifact_paths = sorted(p for p in run_root.rglob('*') if p.is_file()
                            and not p.is_symlink() and p.name != 'evidence_bundle.zip')
    manifest = []
    for artifact in artifact_paths:
        resolved = artifact.resolve()
        if not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact escaped its directory')
        manifest.append({'path': artifact.relative_to(run_root).as_posix(),
                         'bytes': artifact.stat().st_size,
                         'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for artifact in artifact_paths:
            archive.write(artifact, artifact.relative_to(run_root).as_posix())
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    (run_root / 'evidence_bundle.zip').write_bytes(bundle_bytes)
    report = {'sha256': hashlib.sha256(bundle_bytes).hexdigest(), 'bytes': len(bundle_bytes), 'file_count': len(manifest)}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
if IS_KAGGLE and RUN_OFFICIAL_SMOKE:
    report = execute_plan(run, DATA_DIR, MODEL_PATH, RUN, PLAN, checkpoint_exporter=export_run)
    print(json.dumps(report, indent=2))
else:
    print(json.dumps({'status':'not_executed', 'official_leaderboard_score':None}))


## Preserve all assigned outcomes
The next cell exports only this run's actual artifacts. After an infrastructure
exception, run it separately before stopping the worker. Interrupted/unattempted
assignments remain in the denominator; a successful process status cannot turn a
failed official test into a resolved task.


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
from pathlib import Path
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifact_paths = sorted(p for p in run_root.rglob('*') if p.is_file()
                            and not p.is_symlink() and p.name != 'evidence_bundle.zip')
    manifest = []
    for artifact in artifact_paths:
        resolved = artifact.resolve()
        if not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact escaped its directory')
        manifest.append({'path': artifact.relative_to(run_root).as_posix(),
                         'bytes': artifact.stat().st_size,
                         'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for artifact in artifact_paths:
            archive.write(artifact, artifact.relative_to(run_root).as_posix())
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    (run_root / 'evidence_bundle.zip').write_bytes(bundle_bytes)
    report = {'sha256': hashlib.sha256(bundle_bytes).hexdigest(), 'bytes': len(bundle_bytes), 'file_count': len(manifest)}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


if locals().get('IS_KAGGLE', False) and locals().get('RUN_OFFICIAL_SMOKE', False):
    export_run(RUN)
else:
    print('No Kaggle artifacts exported in this local definition check')
